![Kenya Food Price Early Warning System](images/banner.png)

# Kenya Food Price Early Warning System


Forecasting staple food prices across Kenyan markets to give farmers, traders, and food security actors the early signal they currently lack.

## 1. Business Understanding

### 1.1 Background

Food prices in Kenya move sharply and unevenly across markets. Staples such as maize and beans respond to harvest cycles, rainfall, and supply conditions, but the people most exposed to that movement rarely see it coming.

Farmers face a timing decision every season: sell early and risk missing a better price, or hold and risk a drop. Traders and institutions face the same problem from the other side, deciding when to buy, store, or release stock.

None of this is a data problem in the strict sense. Kenya already publishes market price data through WFP and HDX. What is missing is a way to turn that historical record, combined with weather data, into a forward-looking view: where prices are likely headed, and when a market is starting to move outside its own normal range.

### 1.2 Problem Statement

Farmers, traders, and food security institutions largely make decisions on current or historical prices, not forecasts. That gap shows up as poor sell/buy timing, avoidable inventory risk, and institutional responses that arrive only after a shortage or price spike is already visible.

The core problem is not a lack of data. It is the absence of a system that combines historical prices with external drivers like weather to produce forecasts and early warnings at the market level.

### 1.3 Business Objectives

1. **Forward-looking price visibility** — forecast commodity prices two to three months ahead, per market.
2. **Early shock detection** — flag when actual prices start diverging meaningfully from what is expected.
3. **Accessible market intelligence** — surface forecasts and trends through a dashboard usable by non-technical stakeholders.
4. **Decision support at the institutional level** — provide a quantitative signal that can inform reserves, subsidies, procurement, and humanitarian response.
5. **A reproducible pipeline** — one automated path from raw data to forecast to dashboard, not a one-off analysis.

### 1.4 Stakeholder Analysis

**Smallholder farmers and cooperatives** — decide when and how much to sell; forecasts give visibility into where prices are headed before committing.

**Traders and market intermediaries** — manage inventory timing; forecasts reduce the risk of buying near a peak or holding through a decline.

**County agricultural offices and NDMA** — need early, localized signals of price stress before they escalate into broader food security concerns.

**NGOs and humanitarian organizations** — plan procurement and cash-based interventions; earlier visibility protects purchasing power against rising prices.

**National Cereals and Produce Board** — makes reserve, procurement, and stabilization decisions where a forecast is one more input.

**Urban consumers and low-income households** — most exposed to staple price swings; benefit indirectly through institutions that plan ahead on their behalf.

**Food processors and millers** — need predictable input costs for production and pricing decisions.

Across all of these, the shared need is the same: timely, market-specific signal about where food prices are heading, not just where they have been.

### 1.5 Business Success Criteria

This is judged successful from a business standpoint if:

- Forecasts and alerts give stakeholders information they did not already have from watching current prices.
- The anomaly detector surfaces genuine shocks without burying users in false alarms.
- A non-technical user can pick a market and commodity and immediately understand the expected price and current alert status.
- The pipeline can be refreshed with new data without significant manual rework.
- The system runs on public, freely accessible data sources, so it is not tied to a paid feed that could disappear.

### 1.6 Data Mining Goals

The technical work behind those business goals breaks into five tasks:

1. Build a **naive persistence baseline**, since any model must justify itself against it.
2. Build a **Prophet model** as the first real forecasting approach.
3. Build an **LSTM model**, both per-pair and pooled across markets and commodities, for comparison.
4. Join WFP Kenya food price data with NASA POWER weather data by market location and date.
5. Build a **residual-based anomaly detector** to flag unusual price movements.

All of this is meant to run across the full shortlist of market-commodity pairs, not a single illustrative series, since the dashboard needs coverage across markets to be useful. Results are compared using MAE and MAPE, and delivered through a Streamlit dashboard.

### 1.7 Data Mining Success Criteria

This is judged successful from a technical standpoint if:

- Forecast accuracy is measured with MAE and MAPE, consistently, across models.
- A model's usefulness is judged against the **naive persistence baseline**, not an arbitrary fixed error threshold. Beating the baseline is the bar; the baseline's actual value is established once it is computed in Modelling, not assumed here.
- The price and weather datasets join with minimal data loss.
- The anomaly detector shows a meaningfully higher flag rate during documented historical shocks than its own baseline flag rate.
- The pipeline runs end-to-end with minimal manual intervention, and produces the **same result on repeated runs** — which means every stochastic step (model initialization, training) needs to be seeded, not left to chance.
- The deployed dashboard loads reliably and reflects the pipeline's latest output.

### 1.8 Hypotheses Guiding the Analysis

- **H1:** Maize prices follow a seasonal pattern tied to harvest periods.
- **H2:** Rainfall has a measurable lagged relationship with future commodity prices.
- **H3:** Price patterns differ meaningfully across markets.
- **H4:** Maize and bean prices move together, consistent with their role as substitute staples.
- **H5:** Price volatility increases during drought periods.
- **H6:** Wholesale price changes are reflected in retail prices within one to two weeks.

These are tested, not assumed, through the EDA, feature engineering, and modelling sections that follow.

## 2. Data Understanding

This project draws on two datasets: historical food prices from WFP, and daily weather observations from NASA POWER. The goal here is to establish what each dataset contains, how it is structured, and where its quality limits are, before any cleaning or joining takes place.

### 2.1 Source of Data

| Dataset | Source | Purpose |
|---|---|---|
| WFP Kenya Food Prices | HDX (Humanitarian Data Exchange) | Historical commodity prices — the target variable |
| NASA POWER | NASA POWER API | Daily weather observations — external explanatory variables |



In [25]:
# core libraries and notebook display settings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [26]:
# record the date the data snapshot was pulled, for reproducibility
extraction_date = datetime.now().strftime("%Y-%m-%d")
print(f"Data extraction date recorded: {extraction_date}")

Data extraction date recorded: 2026-09-10


In [27]:
DATA_URL = (
    "https://data.humdata.org/dataset/"
    "e0d3fba6-f9a2-45d7-b949-140c455197ff/"
    "resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/"
    "download/wfp_food_prices_ken.csv"
)
FILENAME = "wfp_food_prices_ken.csv"


def load_food_prices():
    """
    Load the Kenya food prices dataset.

    Checks, in order: a Kaggle input copy, a saved local copy,
    then falls back to the source URL.

    Returns
    -------
    pandas.DataFrame
        Raw Kenya food prices dataset.
    """
    kaggle_root = "/kaggle/input"

    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if FILENAME in files:
                kaggle_path = os.path.join(root, FILENAME)
                try:
                    prices_raw = pd.read_csv(kaggle_path)
                    print(f"Loaded dataset from Kaggle: {kaggle_path}")
                    return prices_raw
                except Exception as error:
                    print(f"Kaggle file could not be read: {error}")

    if os.path.exists(FILENAME):
        try:
            prices_raw = pd.read_csv(FILENAME)
            print(f"Loaded local dataset: {FILENAME}")
            return prices_raw
        except Exception as error:
            print(f"Local file could not be read: {error}")

    print("Dataset not found locally. Trying the source URL...")
    try:
        response = requests.get(DATA_URL, timeout=30)
        response.raise_for_status()
        with open(FILENAME, "wb") as file:
            file.write(response.content)
        prices_raw = pd.read_csv(FILENAME)
        print(f"Dataset downloaded and saved as '{FILENAME}'")
        return prices_raw
    except Exception as error:
        print(f"Download failed: {error}")

    if os.path.exists(FILENAME):
        print("Using the previously saved local dataset.")
        return pd.read_csv(FILENAME)

    raise FileNotFoundError(
        "Could not load the Kenya food prices dataset. "
        "Check your internet connection or provide a local copy."
    )

In [28]:
prices_raw = load_food_prices()
prices_raw.head()

Loaded local dataset: wfp_food_prices_ken.csv


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15
3,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Potatoes (Irish),148,50 KG,actual,Wholesale,KES,1249.99,17.39


In [29]:
# drop the units/description row if present, then fix dtypes
def clean_price_data(prices_raw):
    """Clean and convert data types in the raw price dataset."""
    if prices_raw.iloc[0].astype(str).str.startswith("#").any():
        prices = prices_raw.iloc[1:].reset_index(drop=True)
    else:
        prices = prices_raw.copy()

    prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
    prices["price"] = pd.to_numeric(prices["price"], errors="coerce")

    if "usdprice" in prices.columns:
        prices["usdprice"] = pd.to_numeric(prices["usdprice"], errors="coerce")

    return prices


prices = clean_price_data(prices_raw)
print(f"Rows: {prices.shape[0]}, Columns: {prices.shape[1]}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")

Rows: 27763, Columns: 16
Date range: 2006-01-15 00:00:00 to 2026-08-15 00:00:00


In [30]:
# confirm the NASA POWER API is reachable and returns the expected structure
def get_weather_sample(latitude, longitude, start="20240101", end="20240131"):
    """Fetch a short sample of daily weather data for one point."""
    power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M",
        "community": "ag",
        "longitude": longitude,
        "latitude": latitude,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    response = requests.get(power_url, params=params, timeout=30)
    response.raise_for_status()
    print(f"status code {response.status_code}")
    return response.json()["properties"]["parameter"]


nairobi_lat, nairobi_lon = -1.2864, 36.8172
sample_params = get_weather_sample(latitude=nairobi_lat, longitude=nairobi_lon)
print("Parameters returned:", list(sample_params.keys()))

status code 200
Parameters returned: ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']


### 2.2 Dataset Description

The two datasets sit at different grains and are joined on market location and date.

**WFP food prices** — one row per commodity, market, and date. Each row is a single reported price for a specific pairing of item, place, and time, at either retail or wholesale level.

**NASA POWER weather** — one row per day for a given latitude/longitude point. There is no market identifier in the weather data itself; the link back to a market happens through the market's coordinates, which are already present in the price dataset.

This means the join key is not a shared ID column but a derived one: market coordinates plus date, matched to the nearest weather point.

### 2.3 Feature/Data Dictionary

**WFP Kenya Food Prices**

| Column | Type | Description | Example | Role |
|---|---|---|---|---|
| `date` | datetime64 | Date of the price observation | 2006-01-15 | Temporal |
| `admin1` | string | Top-level administrative region | Coast, Eastern | Metadata |
| `admin2` | string | Second-level administrative region | Mombasa, Kitui | Metadata |
| `market` | string | Name of the local food market | Mombasa, Kitui | Identifier / grouping |
| `market_id` | integer | Unique market identifier | 191 | Identifier |
| `latitude` | float | Market latitude | -4.05 | Geographic (join key) |
| `longitude` | float | Market longitude | 39.67 | Geographic (join key) |
| `category` | string | Broad commodity group | cereals and tubers | Metadata |
| `commodity` | string | Specific food item | Maize (white), Beans | Identifier / grouping |
| `commodity_id` | integer | Unique commodity identifier | 67 | Identifier |
| `unit` | string | Unit the price is quoted in | KG, 90 KG | Scoping |
| `priceflag` | string | Source data state | actual | Metadata |
| `pricetype` | string | Trade level | Wholesale, Retail | Scoping |
| `currency` | string | Currency | KES | Metadata |
| **`price`** | float | Price in Kenyan Shillings | 1480.00 | **Target variable (raw)** |
| `usdprice` | float | Price in USD equivalent | 20.58 | Not used |

`price` is the raw form of the target. It is standardized into `price_per_kg` during Data Preparation, once unit and trade-level scoping are resolved, and that standardized value is what the models are actually trained to predict.

**NASA POWER Weather**

| Parameter | Type | Description | Unit | Role |
|---|---|---|---|---|
| `T2M` | float | Average air temperature at 2m | °C | Predictor (used) |
| `T2M_MAX` | float | Maximum air temperature at 2m | °C | Not used |
| `T2M_MIN` | float | Minimum air temperature at 2m | °C | Not used |
| `PRECTOTCORR` | float | Bias-corrected total precipitation | mm/day | Predictor (used) |
| `RH2M` | float | Relative humidity at 2m | % | Not used |

Rainfall (`PRECTOTCORR`) and temperature (`T2M`) are the two variables this project carries forward as predictors, since they have the clearest plausible link to agricultural output and price movement.

### 2.4 Initial Data Quality Assessment

In [31]:
# column-level summary: dtype, uniqueness, missingness
data_dictionary = pd.DataFrame({
    "column": prices.columns,
    "dtype": [str(prices[col].dtype) for col in prices.columns],
    "n_unique": [prices[col].nunique() for col in prices.columns],
    "n_missing": [prices[col].isna().sum() for col in prices.columns],
    "pct_missing": [(prices[col].isna().mean() * 100).round(2) for col in prices.columns],
})
data_dictionary

,column,dtype,n_unique,n_missing,pct_missing
0,date,datetime64[us],248,0,0.00
1,admin1,str,7,62,0.22
2,admin2,str,27,62,0.22
3,market,str,226,0,0.00
4,market_id,int64,226,0,0.00
5,latitude,float64,182,62,0.22
6,longitude,float64,184,62,0.22
7,category,str,8,0,0.00
8,commodity,str,51,0,0.00
9,commodity_id,int64,51,0,0.00


In [32]:
prices[["price", "usdprice"]].describe()

,price,usdprice
count,27763.000000,27763.000000
mean,1308.392522,11.949639
std,2583.636443,22.958342
min,5.000000,0.039000
25%,70.000000,0.610000
50%,130.000000,1.030000
75%,993.000000,9.045000
max,19800.000000,184.820000


In [33]:
grain_columns = ["date", "market", "commodity", "pricetype"]
duplicate_count = prices.duplicated(subset=grain_columns).sum()

print(f"Duplicate rows at (date, market, commodity, pricetype) grain: {duplicate_count}")
print(f"Unique markets: {prices['market'].nunique()}")
print(f"Unique commodities: {prices['commodity'].nunique()}")
print(f"Unique admin1 regions: {prices['admin1'].nunique()}")

Duplicate rows at (date, market, commodity, pricetype) grain: 0
Unique markets: 226
Unique commodities: 51
Unique admin1 regions: 7


In [34]:
# confirm lat/lon exist directly in the price data, and check coverage
coordinate_columns = [col for col in prices.columns if "lat" in col.lower() or "lon" in col.lower()]
coord_check = prices.groupby("market")[coordinate_columns].nunique()

missing_coords = prices[prices["latitude"].isna()]["market"].unique()

print("Coordinate columns found:", coordinate_columns)
print(f"Markets with missing coordinates: {len(missing_coords)}")
print(missing_coords)

Coordinate columns found: ['latitude', 'longitude']
Markets with missing coordinates: 1
<ArrowStringArray>
['Hola (Tana River)']
Length: 1, dtype: str


Latitude and longitude are already present in the price dataset itself, one fixed pair per market, so no separate market-reference file is needed for the weather join.

**Summary of data quality findings:**

- **Commodity labelling:** Maize appears under five separate labels (`Maize (white)`, `Maize`, `Maize flour`, `Maize (white, dry)`, `Maize flour (white)`). These need consolidation before analysis, since treating them as unrelated commodities would understate maize's true market coverage.
- **Duplicates:** none at the (date, market, commodity, pricetype) grain.
- **Coordinates:** complete for 225 of 226 markets. Hola (Tana River) is the sole exception and will be excluded from the weather join rather than imputed.
- **Missingness:** confined to `admin1`, `admin2`, `latitude`, `longitude` — all 62 rows tied to the same single market.
- **Coverage:** the price series spans January 2006 to August 2026, a long enough window to support both long-history and short-history modelling tracks.
- **Units and trade level:** `unit` and `pricetype` are not yet standardized (13 distinct units, retail and wholesale mixed) — this is scoped and resolved explicitly in Data Preparation, not here.

The dataset is fit for the next stage. The open items above are cleanup work, not blockers.

## 3. Data Cleaning and Data Preparation

This phase turns the raw price table into a modelling-ready dataset. Commodity labels are kept as reported rather than merged, since the source data already distinguishes products, such as raw grain from flour, that behave differently in price.

Scope, coverage, and quality decisions are made explicit here so that what enters modelling is a deliberate shortlist, not whatever happened to survive incidental filtering.

### 3.1 Initial Cleaning

Row-level type coercion (parsing `date`, `price`, and `usdprice`) already happened in Data Understanding, since it was needed to profile the data honestly. The first substantive preparation step is reviewing what the `commodity` field actually contains: 51 distinct labels, some of them raw or staple products, others processed derivatives such as flour or meal. A visibility flag distinguishes the two groups. It is not used to merge products together — commodity identity is preserved throughout.

In [35]:
commodity_overview = prices["commodity"].value_counts()
is_processed = prices["commodity"].str.contains("flour|meal|powder", case=False, na=False)
prices["is_processed"] = is_processed

print(f"Total distinct commodity labels: {prices['commodity'].nunique()}")
print(f"Processed or derivative product rows: {is_processed.sum()}")
commodity_overview.head(10)

Total distinct commodity labels: 51
Processed or derivative product rows: 2815


commodity
Beans (dry)         1854
Maize (white)       1656
Maize               1518
Sugar               1401
Beans               1386
Wheat flour         1380
Salt                1362
Potatoes (Irish)    1320
Sorghum             1021
Rice (aromatic)     1017
Name: count, dtype: int64

### 3.2 Data Type Handling

`price` is only meaningful once its unit is known — 1,480 KES means something different for a kilogram of maize than for a 90 kilogram sack. Every distinct unit in the dataset is mapped to a kilogram-equivalent multiplier, covering all weight-based units observed, not just the ones maize happens to use.

In [36]:
unit_counts = prices["unit"].value_counts()
print(unit_counts)

unit
KG        14595
90 KG      4118
L          2715
50 KG      1905
200 G      1362
500 ML      804
Unit        591
64 KG       518
13 KG       470
126 KG      459
400 G       178
Head         26
Bunch        22
Name: count, dtype: int64


In [37]:
unit_to_kg = {
    "KG": 1,
    "90 KG": 90,
    "64 KG": 64,
    "50 KG": 50,
    "26 KG": 26,
    "126 KG": 126,
    "13 KG": 13,
    "200 G": 0.2,
    "400 G": 0.4,
}

prices["kg_equivalent"] = prices["unit"].map(unit_to_kg)

### 3.3 Unit Standardization

With a kilogram-equivalent defined for every weight-based unit, price becomes directly comparable across records as `price_per_kg`. Units with no weight equivalent — sold by volume or by count — are left unconverted rather than force-fit, and identified explicitly here so the next step can make a clean scope decision about them.

In [38]:
prices["price_per_kg"] = prices["price"] / prices["kg_equivalent"]

unmapped_units = prices[prices["kg_equivalent"].isna()]["unit"].unique()
print(f"Unmapped units, genuinely non-weight based: {list(unmapped_units)}")
print(f"Rows converted to price per kg: {prices['price_per_kg'].notna().sum()} out of {len(prices)}")

Unmapped units, genuinely non-weight based: ['500 ML', 'L', 'Unit', 'Bunch', 'Head']
Rows converted to price per kg: 23605 out of 27763


### 3.4 Commodity and Scope Selection

Two categories of commodity are removed before completeness or coverage is ever computed, rather than left to fail silently at the price-per-kg step several sections later.

Fuel (diesel, kerosene, petrol-gasoline) is out of scope — this is a food price project, and these three are present only because the WFP dataset tracks a broader commodity basket than food. Commodities priced in a unit with no kilogram equivalent — litres, millilitres, or a bare count — are excluded on measurement grounds, not as a data quality failure. Milk in all four varieties, vegetable oil, bananas, kale, and cabbage fall into this group: legitimate food commodities, but priced in a way a per-kilogram index was never built to handle. Filtering by unit compatibility rather than hardcoding these names also protects against any other commodity sharing the same problem.

In [39]:
FUEL_COMMODITIES = ["Fuel (diesel)", "Fuel (kerosene)", "Fuel (petrol-gasoline)"]

excluded_units = sorted(set(prices["unit"]) - set(unit_to_kg))
unit_excluded_commodities = prices[prices["unit"].isin(excluded_units)]["commodity"].unique()

print(f"Excluding {len(FUEL_COMMODITIES)} fuel commodities on scope grounds")
print(f"Excluding {len(unit_excluded_commodities)} commodities priced in a non-weight unit: {list(unit_excluded_commodities)}")

prices_scoped = prices[
    ~prices["commodity"].isin(FUEL_COMMODITIES) &
    prices["unit"].isin(unit_to_kg)
].copy()

print(f"Rows after scope filtering: {len(prices_scoped)} out of {len(prices)}")

Excluding 3 fuel commodities on scope grounds
Excluding 12 commodities priced in a non-weight unit: ['Milk (cow, pasteurized)', 'Oil (vegetable)', 'Fuel (diesel)', 'Fuel (kerosene)', 'Fuel (petrol-gasoline)', 'Bananas', 'Kale', 'Oil (vegetable, fortified)', 'Milk (UHT)', 'Cabbage', 'Milk (camel, fresh)', 'Milk (cow, fresh)']
Rows after scope filtering: 23605 out of 27763


### 3.5 Retail Price Selection

Retail is selected as the primary modelling series — a deliberate scoping decision, not an oversight. Retail is the price point that smallholder farmers, cooperatives, and urban consumers, the majority of the stakeholder groups from Section 1, actually transact on. Wholesale forecasting, more relevant to traders, NCPB, and millers, is a natural extension of the same pipeline: wholesale rows remain intact in the source data and need no new collection, only rerunning completeness and modelling against `pricetype == "Wholesale"`. That is documented here as a defined next phase, not an unaddressed gap.

In [40]:
retail = prices_scoped[prices_scoped["pricetype"] == "Retail"].copy()
wholesale_rows = prices_scoped[prices_scoped["pricetype"] == "Wholesale"]

print(f"Retail rows: {len(retail)}")
print(f"Wholesale rows: {len(wholesale_rows)}")

Retail rows: 14450
Wholesale rows: 9155


### 3.6 Missing Data / Completeness

Completeness is computed for every market-commodity pair at once, against the full months available in the retail dataset, before any threshold is chosen. This gives an honest picture of coverage before the shortlist is narrowed.

In [41]:
coverage = (
    retail
    .groupby(["market", "commodity"])["date"]
    .nunique()
    .reset_index(name="months_reported")
)

total_months_available = retail["date"].nunique()
coverage["completeness_pct"] = (coverage["months_reported"] / total_months_available * 100).round(1)

print(f"Total market-commodity pairs: {len(coverage)}")
for threshold in [30, 40, 50, 60, 70]:
    qualifying = coverage[coverage["completeness_pct"] >= threshold]
    print(f"At {threshold}% threshold: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

Total market-commodity pairs: 2021
At 30% threshold: 6 pairs, 5 markets, 3 commodities
At 40% threshold: 6 pairs, 5 markets, 3 commodities
At 50% threshold: 5 pairs, 4 markets, 3 commodities
At 60% threshold: 4 pairs, 3 markets, 3 commodities
At 70% threshold: 4 pairs, 3 markets, 3 commodities


Measured against the full dataset span, almost nothing qualifies. That is because most series only began consistent reporting in late 2023 — a fixed-span measurement unfairly penalizes pairs that started later but have reported reliably since. Coverage needs to be measured against each pair's *own* active reporting window instead.

In [42]:
reporting_span = (
    retail
    .groupby(["market", "commodity"])["date"]
    .agg(first_reported="min", last_reported="max", months_reported="nunique")
    .reset_index()
)

reporting_span["active_months"] = (
    (reporting_span["last_reported"].dt.year - reporting_span["first_reported"].dt.year) * 12
    + (reporting_span["last_reported"].dt.month - reporting_span["first_reported"].dt.month)
    + 1
)

reporting_span["completeness_pct_fair"] = (
    reporting_span["months_reported"] / reporting_span["active_months"] * 100
).round(1)

for threshold in [50, 60, 70, 80, 90]:
    qualifying = reporting_span[reporting_span["completeness_pct_fair"] >= threshold]
    print(f"At {threshold}% fair completeness: {len(qualifying)} pairs, {qualifying['market'].nunique()} markets, {qualifying['commodity'].nunique()} commodities")

At 50% fair completeness: 826 pairs, 176 markets, 25 commodities
At 60% fair completeness: 675 pairs, 161 markets, 25 commodities
At 70% fair completeness: 591 pairs, 159 markets, 25 commodities
At 80% fair completeness: 524 pairs, 153 markets, 25 commodities
At 90% fair completeness: 499 pairs, 149 markets, 25 commodities


### 3.7 Historical Coverage

Sufficient observations are not the same as sufficient history. Detecting a genuine seasonal pattern requires seeing it repeat across multiple years, so pairs are classified by both fair completeness and years of active history. Deeper-history pairs are routed to Prophet, which needs multiple seasonal cycles to be reliable; shorter but still-consistent pairs are routed to LSTM instead of being discarded.

In [43]:
reporting_span["years_active"] = (
    (reporting_span["last_reported"] - reporting_span["first_reported"]).dt.days / 365.25
).round(1)

MIN_YEARS_ACTIVE = 1.0

long_history = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= 3)
].copy()

recent_only = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] >= MIN_YEARS_ACTIVE) &
    (reporting_span["years_active"] < 3)
].copy()

insufficient_data = reporting_span[
    (reporting_span["completeness_pct_fair"] >= 60) &
    (reporting_span["years_active"] < MIN_YEARS_ACTIVE)
]

print(f"Long history pairs (3+ years): {len(long_history)}")
print(f"Recent only pairs (1 to 3 years): {len(recent_only)}")
print(f"Insufficient data pairs (under 1 year, excluded): {len(insufficient_data)}")

Long history pairs (3+ years): 100
Recent only pairs (1 to 3 years): 33
Insufficient data pairs (under 1 year, excluded): 542


Together, the long-history and recent-only groups define the full modelling scope. Each pair keeps its original commodity label, market, and price type — nothing is merged.

In [44]:
long_history["model_track"] = "prophet"
recent_only["model_track"] = "lstm"

shortlist = pd.concat([long_history, recent_only], ignore_index=True)[["market", "commodity", "model_track"]]

print(f"Total shortlisted market-commodity pairs: {len(shortlist)}")
print(f"Unique markets: {shortlist['market'].nunique()}")
print(f"Unique commodities: {shortlist['commodity'].nunique()}")

Total shortlisted market-commodity pairs: 133
Unique markets: 26
Unique commodities: 14


In [45]:
retail["kg_equivalent"] = retail["unit"].map(unit_to_kg)
retail["price_per_kg"] = retail["price"] / retail["kg_equivalent"]

modeling_data = retail.merge(shortlist, on=["market", "commodity"], how="inner")
modeling_data = modeling_data[modeling_data["price_per_kg"].notna()].copy()

print(f"Modelling-ready rows: {len(modeling_data)}")

Modelling-ready rows: 6077


### 3.8 Outlier Handling

Outlier bounds are computed on the standardized `price_per_kg` field, separately per commodity — pooling different commodities together would produce meaningless bounds, the same mistake an unstandardized check would make.

In [46]:
def flag_outliers(group):
    q1, q3 = group["price_per_kg"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return group[(group["price_per_kg"] < lower) | (group["price_per_kg"] > upper)]

outliers_by_commodity = (
    modeling_data
    .groupby("commodity", group_keys=True)
    .apply(flag_outliers, include_groups=False)
    .reset_index(level=0)
)

print(f"Total outliers flagged: {len(outliers_by_commodity)} out of {len(modeling_data)}")
print(outliers_by_commodity["commodity"].value_counts())

Total outliers flagged: 69 out of 6077
commodity
Meat (camel)        26
Salt                17
Beans (dry)          8
Maize                5
Maize (white)        4
Rice                 3
Potatoes (Irish)     2
Sugar                2
Meat (beef)          1
Sorghum              1
Name: count, dtype: int64


Manual inspection of the salt outliers shows a tight, plausible cluster around 90–115 KES/kg, with a tail toward 275 KES/kg concentrated in refugee-camp markets such as Kakuma, Dadaab, and Kalobeyei, where transport and supply chain costs run higher. These are genuine prices, not conversion errors — and every flagged outlier, across every commodity, is retained rather than removed, since the anomaly detection layer built later is designed specifically to act on this kind of divergence.

In [47]:
salt_outliers = outliers_by_commodity[outliers_by_commodity["commodity"] == "Salt"]
salt_outliers[["market", "date", "price", "unit", "price_per_kg"]]

,market,date,price,unit,price_per_kg
1502,Kakuma 4,2025-09-15,20.00,200 G,100.00
1508,Ethiopia (Kakuma),2025-12-15,20.00,200 G,100.00
1511,HongKong (Kakuma),2025-12-15,23.00,200 G,115.00
1516,Kakuma 3,2025-12-15,20.00,200 G,100.00
1519,Kakuma 4,2025-12-15,18.00,200 G,90.00
1570,IFO (Daadab),2024-06-15,39.26,200 G,196.30
1741,Kakuma 4,2026-03-15,16.50,200 G,82.50
3723,Kalobeyei (Village 1),2023-07-15,49.38,200 G,246.90
4410,IFO (Daadab),2024-05-15,35.98,200 G,179.90
5400,Kakuma 3,2025-08-15,55.00,200 G,275.00


An IQR flag alone can't tell a genuine market shift, where prices settle at a new level and stay there, from a transient spike that reverts, or a one-off entry error. Each flagged outlier is classified by comparing the price level immediately after the flagged date against the baseline immediately before it. This same before/after comparison logic is the direct precursor to the residual-based anomaly detector built in Section 7.

In [48]:
def classify_outlier(row, data, window=2):
    series = data[(data["market"] == row["market"]) & (data["commodity"] == row["commodity"])].sort_values("date")
    match = series[series["date"] == row["date"]]
    if match.empty:
        return "unknown"
    pos = series.index.get_loc(match.index[0])
    before = series.iloc[max(0, pos - window):pos]["price_per_kg"]
    after = series.iloc[pos + 1: pos + 1 + window]["price_per_kg"]
    if before.empty or after.empty:
        return "insufficient surrounding data"
    baseline = before.mean()
    reverted = abs(after.mean() - baseline) < abs(row["price_per_kg"] - baseline) * 0.5
    return "transient spike" if reverted else "persistent shift"

outliers_by_commodity["classification"] = outliers_by_commodity.apply(
    lambda row: classify_outlier(row, modeling_data), axis=1
)

print(outliers_by_commodity["classification"].value_counts())

classification
persistent shift                 45
transient spike                  22
insufficient surrounding data     2
Name: count, dtype: int64


No flagged outlier is removed from the modelling dataset. Transient spikes are retained as genuine historical events, reserved as informal validation cases for the anomaly detector built later. Persistent shifts are retained too, though a sample was manually reviewed to rule out unit or entry errors masquerading as real shifts, since the two produce an identical pattern on this test. Rows with insufficient surrounding data are kept but not treated as evidence either way.

### 3.9 Dataset Integration

With the shortlist finalized, daily rainfall and temperature are retrieved from the NASA POWER API for every shortlisted market's coordinates, across the full date range needed for modelling. Weather is recorded daily but price is recorded monthly, so weather is aggregated to a monthly grain per market before joining — rainfall summed, temperature averaged, matching how each variable naturally accumulates over a month.

In [50]:
import time

market_coords = modeling_data[["market", "latitude", "longitude"]].drop_duplicates()
weather_records = []
power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"




for _, row in market_coords.iterrows():
    params_loop = {
        "parameters": "T2M,PRECTOTCORR",
        "community": "ag",
        "longitude": row["longitude"],
        "latitude": row["latitude"],
        "start": "20060101",
        "end": "20260815",
        "format": "JSON",
    }
    resp = requests.get(power_url, params=params_loop, timeout=60)
    if resp.status_code == 200:
        param_data = resp.json()["properties"]["parameter"]
        df_market = pd.DataFrame({
            "date": pd.to_datetime(list(param_data["T2M"].keys()), format="%Y%m%d"),
            "temperature": list(param_data["T2M"].values()),
            "rainfall": list(param_data["PRECTOTCORR"].values()),
        })
        df_market["market"] = row["market"]
        weather_records.append(df_market)
    time.sleep(1)

weather_all = pd.concat(weather_records, ignore_index=True)

print(f"Markets retrieved: {weather_all['market'].nunique()} out of {len(market_coords)}")
print(f"Total daily weather rows: {len(weather_all)}")

Markets retrieved: 25 out of 26
Total daily weather rows: 188300


In [51]:
weather_monthly = (
    weather_all
    .set_index("date")
    .groupby("market")
    .resample("ME")
    .agg({"rainfall": "sum", "temperature": "mean"})
    .reset_index()
)

print(f"Monthly weather rows: {len(weather_monthly)}")
print(f"Markets represented: {weather_monthly['market'].nunique()}")

Monthly weather rows: 6200
Markets represented: 25


Rainfall and temperature are also shifted forward by three and four months per market here, so no market's lag mixes with another market's history. These lagged columns travel with the master table into Exploratory Data Analysis, where their relationship with price is actually tested, and are only finalized as modelling inputs in Feature Engineering.

In [52]:
weather_monthly = weather_monthly.sort_values(["market", "date"])
weather_monthly["rainfall_lag_3"] = weather_monthly.groupby("market")["rainfall"].shift(3)
weather_monthly["rainfall_lag_4"] = weather_monthly.groupby("market")["rainfall"].shift(4)
weather_monthly["temperature_lag_3"] = weather_monthly.groupby("market")["temperature"].shift(3)
weather_monthly["temperature_lag_4"] = weather_monthly.groupby("market")["temperature"].shift(4)

Cleaned monthly prices are joined to the lagged monthly weather on market and date. The weather table's own `date` column is dropped before merging — both tables otherwise carry a column named `date`, which would silently become `date_x`/`date_y` and break every downstream step that references `date` directly.

In [53]:
modeling_data["date_month"] = modeling_data["date"].values.astype("datetime64[M]")
weather_monthly["date_month"] = weather_monthly["date"].values.astype("datetime64[M]")
weather_features = weather_monthly.drop(columns=["date"])

master = modeling_data.merge(
    weather_features,
    on=["market", "date_month"],
    how="left"
)

print(f"Master table rows: {len(master)}")
print(f"Rows with matched weather data: {master['rainfall'].notna().sum()}")

Master table rows: 6077
Rows with matched weather data: 6015


### 3.10 Data Preparation Summary

Three structural decisions shaped this phase. Commodity labels were kept distinct rather than merged, since WFP records raw grain and flour, for instance, as genuinely different products with different price behavior. Completeness was measured against each pair's own active reporting window rather than the full dataset span, since most series only began consistent reporting in late 2023, and a fixed-span measure would have unfairly penalized everything that started later. Units were standardized to a common price-per-kilogram basis, with weight-based units converted and genuinely non-weight units left out of weight-based analysis entirely.

The result is a shortlist of 133 market-commodity pairs across 26 markets and 14 commodities: 100 pairs with three or more years of history routed to Prophet, and 33 shorter-history pairs routed to LSTM. Outliers were identified per commodity and retained rather than removed, since the anomaly detection layer is designed to act on genuine divergence, not treat it as noise. Weather was retrieved, aggregated to monthly, lagged by three and four months, and joined to price, producing a master table of 6,077 rows with a 98.98% weather match rate — the one shortlisted market without a match, Hola (Tana River), lacks coordinates in the source data.

Whether that three- and four-month rainfall lag actually holds up is tested properly in Exploratory Data Analysis, once weather and price share a genuinely matched market and date for every row.